In [1]:
from chonkie import TokenChunker

chunker = TokenChunker()

text= """Woah! Chonkie, the chunking library is so cool!"""

chunks=chunker(text)
for chunk in chunks:
    print(f" Chunk : {chunk.text}")
    print(f" Tokens : {chunk.token_count}")


 Chunk : Woah! Chonkie, the chunking library is so cool!
 Tokens : 47


### Logging

In [ ]:
# export CHONKIE_LOG=debug  # Show everything
# export CHONKIE_LOG=info   # Show info, warnings, and errors
# export CHONKIE_LOG=warning # Show warnings and errors (default)
# export CHONKIE_LOG=error  # Show only errors
# export CHONKIE_LOG=off    # Disable all logging

In [2]:
import chonkie

# Enable debug logging
chonkie.logger.configure("debug")

# # Disable logging completely
# chonkie.logger.configure("off")

# # Back to warnings and errors
# chonkie.logger.configure("warning")

In [3]:
import chonkie
import logging

# Custom formatter that includes extra fields
class StructuredFormatter(logging.Formatter):
    def format(self, record):
        msg = super().format(record)
        # Add any extra fields
        extras = []
        for key in ['chunk_count', 'tokens']:
            if hasattr(record, key):
                extras.append(f"{key}={getattr(record, key)}")
        if extras:
            msg += f" [{', '.join(extras)}]"
        return msg

# Apply custom formatter
logger = logging.getLogger("chonkie")
handler = logging.StreamHandler()
handler.setFormatter(StructuredFormatter("%(levelname)s - %(message)s"))
logger.addHandler(handler)
logger.setLevel(logging.INFO)

### Building Pipelines

Chonkie’s Pipeline API provides a fluent, chainable interface for building text processing workflows. Pipelines follow the CHOMP architecture, automatically orchestrating components in the correct order.

What is CHOMP?
CHOMP (CHOnkie’s Multi-step Pipeline) is our standardized architecture for document processing:

    Fetcher → Chef → Chunker → Refinery → Porter/Handshake

1. Fetcher
Retrieve raw data from files, APIs, or databases
2. Chef
Preprocess and transform raw data into Documents
3. Chunker
Split documents into manageable chunks
4. Refinery (Optional)
Post-process and enhance chunks
5. Porter/Handshake (Optional)
Export or store chunks

#### Single File Processing

In [7]:
from chonkie import Pipeline

doc = (Pipeline()
       .fetch_from("file", path="document.txt")
       .process_with("text")
       .chunk_with("recursive", chunk_size=200)
       .run()
       )

print(f"Created {len(doc.chunks)} chunks")

for chunk in doc.chunks:
    print(f"CHunk : {chunk.text[:50]}...")

2026-05-15 13:23:06,937 | INFO     | chonkie.chef.text:process:32 - Text processing complete: read 2042 characters from document.txt
INFO - Text processing complete: read 2042 characters from document.txt
2026-05-15 13:23:06,939 | INFO     | chonkie.chunker.recursive:chunk:238 - Created 15 chunks using recursive chunking
INFO - Created 15 chunks using recursive chunking


Created 15 chunks
CHunk : In deep learning, the transformer is a family of a...
CHunk :  in which text is converted to numerical represent...
CHunk :  each token is then contextualized within the scop...
CHunk :  allowing the signal for key tokens to be amplifie...
CHunk : Because self-attention alone is permutation-invari...
CHunk :  so token order can affect the output.[2]

...
CHunk : Transformers have the advantage of having no recur...
CHunk :  Later variations have been widely adopted for tra...
CHunk :  Modern transformer designs are commonly grouped i...
CHunk :  autoregressive generation, or conditional sequenc...
CHunk : The original version of the transformer architectu...
CHunk :  The predecessors of transformers were developed a...
CHunk : They are used in large-scale natural language proc...
CHunk :  and at disaster responses[13]. ...
CHunk : It has also led to the development of pre-trained ...


#### Directory Processing

In [ ]:
# Process all markdown files in a directory
docs = (Pipeline()
    .fetch_from("file", dir="./documents", ext=[".md", ".txt"])
    .process_with("text")
    .chunk_with("recursive", chunk_size=512)
    .run())

# Process each document
for doc in docs:
    print(f"Document has {len(doc.chunks)} chunks")

#### Direct Text Input

In [16]:
# No fetcher needed
doc = (Pipeline()
    .process_with("text")
    .chunk_with("semantic", threshold=0.8)
    .run(texts="In deep learning, the transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, in which text is converted to numerical representations called tokens, and each token is converted into a vector via lookup from a word embedding table.[1] At each layer, each token is then contextualized within the scope of the context window with other (unmasked) tokens via a parallel multi-head attention mechanism, allowing the signal for key tokens to be amplified and less important tokens to be diminished. Because self-attention alone is permutation-invariant, transformers inject positional information, typically through positional encodings or learned positional embeddings, so token order can affect the output.[2]"))
print(doc.chunks)
# Multiple texts
docs = (Pipeline()
    .chunk_with("recursive", chunk_size=512)
    .run(texts=["Memory is a system that remembers information about previous interactions. For AI agents, memory is crucial because it lets them remember previous interactions, learn from feedback, and adapt to user preferences. As agents tackle more complex tasks with numerous user interactions, this capability becomes essential for both efficiency and user satisfaction.",
                "Conversation history is the most common form of short-term memory. Long conversations pose a challenge to today’s LLMs; a full history may not fit inside an LLM’s context window, resulting in an context loss or errors.",
                "Even if your model supports the full context length, most LLMs still perform poorly over long contexts. They get “distracted” by stale or off-topic content, all while suffering from slower response times and higher costs."]))

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 164482.51it/s]
2026-05-15 13:51:22,940 | INFO     | chonkie.chunker.recursive:chunk:238 - Created 1 chunks using recursive chunking
INFO - Created 1 chunks using recursive chunking
2026-05-15 13:51:22,941 | INFO     | chonkie.chunker.recursive:chunk:238 - Created 1 chunks using recursive chunking
INFO - Created 1 chunks using recursive chunking
2026-05-15 13:51:22,941 | INFO     | chonkie.chunker.recursive:chunk:238 - Created 1 chunks using recursive chunking
INFO - Created 1 chunks using recursive chunking


[Chunk(text='In deep learning, the transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, in which text is converted to numerical representations called tokens, and each token is converted into a vector via lookup from a word embedding table.[1] At each layer, each token is then contextualized within the scope of the context window with other (unmasked) tokens via a parallel multi-head attention mechanism, allowing the signal for key tokens to be amplified and less important tokens to be diminished. Because self-attention alone is permutation-invariant, transformers inject positional information, typically through positional encodings or learned positional embeddings, so token order can affect the output.[2]', token_count=141, start_index=0, end_index=761)]


In [17]:
# Process each document
for doc in docs:
    print(f"Document has {len(doc.chunks)} chunks")



Document has 1 chunks
Document has 1 chunks
Document has 1 chunks


#### Asynchronous Execution

In [18]:
import asyncio

async def process_docs():
    pipe = Pipeline().chunk_with("recursive")

    # Run pipeline asynchronously
    doc = await pipe.arun(texts="Async processing is fast!")

    # Process multiple concurrently
    docs = await pipe.arun(texts=["Doc 1", "Doc 2"])

    return docs

In [21]:
process_docs()

<coroutine object process_docs at 0x75c8a5aa26c0>